In [3]:
import kfp
from kfp import dsl
from kfp import kubernetes
from kfp import local
from kfp.dsl import Input, Output, Dataset, Model, Artifact

# TIP: you may need to authenticate with the KFP instance
# local.init(runner=local.SubprocessRunner())
kfp_client = kfp.Client()

/opt/conda/lib/python3.11/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


In [4]:
current_sc = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.spec.storageClassName}'").read()
namespace_cur = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.metadata.namespace}'").read()
print(namespace_cur)
print(current_sc)

geun-tak-roh-2e590eb8
gl4f-filesystem


In [ ]:
import os
import base64
import kserve
from kubernetes import client,config,utils
from kubernetes.client.rest import ApiException

config.load_incluster_config()
v1 = client.CoreV1Api()
custom_api = client.CustomObjectsApi()

secret_name = 'my-secret'
namespace = namespace_cur

with open('/etc/secrets/ezua/.auth_token','r') as file:
    AUTH_TOKEN = file.read().strip()

try:
    ## Check pre-exists secret
    preexists_secret = v1.read_namespaced_secret(namespace=namespace,name=secret_name)
    print(f"secret name : {preexists_secret.metadata.name} exists")
    v1.delete_namespaced_secret(namespace=namespace,name=secret_name)

except ApiException as e:
    print("Exception when calling CoreV1Api : %s\n" % e)

finally:
    ## Create secret snippet start
    secret_data_encoded = {
        "AWS_ACCESS_KEY_ID": base64.b64encode(AUTH_TOKEN.encode()).decode(),
        "AWS_SECRET_ACCESS_KEY": base64.b64encode("s3".encode()).decode(),
        "AWS_REGION": base64.b64encode("local".encode()).decode(),
    }
    
    secret_body = client.V1Secret(
        api_version="v1",
        kind="Secret",
        metadata=client.V1ObjectMeta(
            name=secret_name,
            annotations={
                "serving.kserve.io/s3-cabundle":"",
                "serving.kserve.io/s3-endpoint":"local-s3-service.ezdata-system.svc.cluster.local:30000/",
                "serving.kserve.io/s3-useanoncredential":"false",
                "serving.kserve.io/s3-usehttps":"0",
                "serving.kserve.io/s3-verifyssl":"0",
            }
        ),
        type="Opaque",
        data=secret_data_encoded,  # Use data or string_data
    )
    
    api_response = v1.create_namespaced_secret(namespace=namespace, body=secret_body)
    print(f"Secret '{api_response.metadata.name}' created successfully in namespace '{namespace}'.\n api_response:{api_response}")
    ## Create secret snippet end

In [6]:
@dsl.component(
    base_image='geuntakroh/kfp-test:v0.9',
)
def write_artifacts(output_artifacts: Output[Artifact]):
    import subprocess
    subprocess.run(['env'])
    print(output_artifacts.path)
    with open(output_artifacts.path,'w') as output_file:
        output_file.write("test artifacts")

In [9]:
@dsl.pipeline(
    name="prepare_model_pipe"
)
def prepare_model_pipe():
    task1 = write_artifacts()
    kfp.kubernetes.use_secret_as_env(
        task=task1,
        secret_name='my-secret',
        secret_key_to_env={'AWS_SECRET_ACCESS_KEY': 'AWS_SECRET_ACCESS_KEY'}
    )
    kubernetes.use_secret_as_env(
        task=task1,
        secret_name='my-secret',
        secret_key_to_env={'AWS_ACCESS_KEY_ID': 'AWS_ACCESS_KEY_ID'}
    )
    kubernetes.use_secret_as_env(
        task=task1,
        secret_name='my-secret',
        secret_key_to_env={'AWS_REGION': 'AWS_REGION'}
    )
    task1.set_env_variable(
        name='AWS_ENDPOINT_URL_S3',
        value='http://local-s3-service.ezdata-system.svc.cluster.local:30000'
    )

In [10]:
kfp_client.create_run_from_pipeline_func(
    prepare_model_pipe,
    experiment_name="test-rhgt-exp",
    enable_caching=False,
    pipeline_root=''
)

RunPipelineResult(run_id=7cf428c7-a29f-46e9-a5f5-34ad4b47b533)

In [11]:
import boto3
import os

In [12]:
url="http://local-s3-service.ezdata-system.svc.cluster.local:30000"
# url="http://pcai-nfs-vip-pool.pcai0103.sy6.hpecolo.net"

In [13]:
s3 = boto3.client('s3',endpoint_url=url)


In [14]:
res = s3.list_buckets()
for i in res['Buckets']:
    print(i)

{'Name': 'mlflow.sg2pcai172', 'CreationDate': datetime.datetime(2025, 8, 4, 7, 49, 26, tzinfo=tzlocal())}


In [15]:
bucket_name = 'mlflow.sg2pcai172'

In [16]:
response = s3.list_objects_v2(Bucket=bucket_name)

In [20]:
response['Contents'][-1]

{'Key': 'v2/artifacts/prepare-model-pipe/7cf428c7-a29f-46e9-a5f5-34ad4b47b533/write-artifacts/output_artifacts',
 'LastModified': datetime.datetime(2025, 8, 4, 7, 49, 26, 831740, tzinfo=tzlocal()),
 'ETag': '"79765f6bc0959d32df00ab31254bb3ad"',
 'Size': 14,
 'StorageClass': 'STANDARD'}

In [21]:
key = response['Contents'][-1]['Key']

In [22]:
key

'v2/artifacts/prepare-model-pipe/7cf428c7-a29f-46e9-a5f5-34ad4b47b533/write-artifacts/output_artifacts'

In [23]:
s3.download_file(bucket_name,key,'./output_artifacts')

In [24]:
!cat output_artifacts

test artifacts